In [1]:
import os
from tqdm import tqdm
from os.path import join
from glob import glob
from PIL import Image
import pandas as pd
import os
from os.path import basename
import numpy as np
import torch
from torchvision import transforms

dataset_path = "/kaggle/input/lost-in-the-museum-v2/archive/kaggle_dataset/kaggle_dataset"

all_images = sorted(glob(join(dataset_path, "*.png")))
print(f"All images: {len(all_images)}")

All images: 20000


In [2]:
# from PIL import Image
# from tqdm import tqdm

# def is_corner_white(image_path, threshold=250):
#     img = Image.open(image_path).convert("RGB")
#     width, height = img.size

#     # Получаем пиксели углов
#     corners = [
#         img.getpixel((0, 0)),                 # верхний левый
#         img.getpixel((width - 1, 0)),         # верхний правый
#         img.getpixel((0, height - 1)),        # нижний левый
#         img.getpixel((width - 1, height - 1)) # нижний правый
#     ]

#     # Проверяем, что хотя бы один угол белый
#     for r, g, b in corners:
#         if r >= threshold and g >= threshold and b >= threshold:
#             return True
#     return False
    
# def check_images(image_paths):
#     """
#     Проверяет список изображений и возвращает словарь:
#     {путь_к_файлу: True/False}
#     """
#     count = 0
#     results = {}
#     for i, path in tqdm(enumerate(image_paths), total = len(image_paths)):
#         results[path] = is_corner_white(path)
#         count += int(results[path])
#         if i % 1000 == 0:
#             print(count)
#     return results


# image_paths = all_images.copy()
# results = check_images(image_paths)

In [3]:
# import matplotlib.pyplot as plt
# for i, el in results.items():
#     if el:
#         plt.imshow(Image.open(i))
#         plt.axis('off')
#         plt.show()

# Siglip

In [4]:
!pip install transformers==4.57.1 -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 89.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 91.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.1.1 requires pyarrow>=21.0.0, but you have pyarrow 19.0.1 which is incompatible.
gradio 5.38.1 requires pydantic<2.12,>=2.0, but you have pydantic 2.12.0a1 which is incompatible.


In [5]:
import torch
from PIL import Image
from transformers import SiglipProcessor, SiglipModel
from tqdm import tqdm
from os.path import basename

def load_image(image_path: str):
    """Load a single image."""
    try:
        image = Image.open(image_path).convert("RGB")
        return image
    except Exception as e:
        print(f"Error loading image {image_path}: {e}")
        return None


def extract_features(image_list: list, device: str = "cuda"):
    """
    Extract cosine-space embeddings using a pre-trained SigLIP model.
    Returns: valid_image_names, features_list
    """
    # 🧠 Load SigLIP
    model_name = "google/siglip2-giant-opt-patch16-256"
    model = SiglipModel.from_pretrained(model_name).to(device)
    processor = SiglipProcessor.from_pretrained(model_name)
    model.eval()

    features_list = []
    valid_image_paths = []

    with torch.no_grad():
        for image_path in tqdm(image_list, desc="Extracting SigLIP features"):
            image = load_image(image_path)
            if image is None:
                continue

            # 🔧 Preprocess (processor делает resize/crop/normalize автоматически)
            inputs = processor(images=image, return_tensors="pt").to(device)

            # 🚀 Forward pass — SigLIP сам возвращает image embeddings
            outputs = model.get_image_features(**inputs)

            # ✨ L2-нормализация (рекомендуется, если не делать потом)
            embeddings = outputs / outputs.norm(p=2, dim=-1, keepdim=True)

            # Move to CPU & numpy
            features_list.append(embeddings[0].cpu().numpy())
            valid_image_paths.append(basename(image_path))

    return valid_image_paths, features_list


2025-10-31 05:32:40.160744: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761888760.342632      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761888760.394005      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [6]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# Use appropriate device type in accordance to your your feature extractor algorithm
valid_paths, features_list = extract_features(all_images, device="cuda") 

output_csv_path = "/kaggle/working/submission_siglip_GIANT.csv"
if not features_list:
    print("Error: No images were successfully processed.")
print(f"Successfully processed {len(features_list)} images")

# Create DataFrame with features
N = max(features_list[0].shape)
column_names = [f"feature_{i}" for i in range(N)]
df = pd.DataFrame(features_list, columns=column_names)

# Add image path column
df.insert(0, 'image_name', valid_paths)
df['ID'] = df['image_name']

# Save to CSV
print(f"Saving features to: {output_csv_path}")
df.to_csv(output_csv_path, index=False)

print(f"Features successfully saved to {output_csv_path}")

config.json:   0%|          | 0.00/537 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.49G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

...

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/34.4M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

Extracting SigLIP features: 100%|██████████| 20000/20000 [50:17<00:00,  6.63it/s]


Successfully processed 20000 images
Saving features to: /kaggle/working/submission_siglip_GIANT.csv
Features successfully saved to /kaggle/working/submission_siglip_GIANT.csv


In [7]:
# ! echo '{"username":"kpmpouifniufipud","key":"34b0fc237087bd931d10c9268931cea7"}' > /root/.config/kaggle/kaggle.json

In [8]:
# !kaggle competitions submit -c lost-in-the-museum-v2 -f submission.csv -m "siglip2GIANT"